In [1]:
from function_library import *

In [ ]:
# Load datasets from specified file paths
datenpfad = 'D:/AG_USB/Mohamed_AG_Schock/Datasets/lmu_big_last.csv'
df_AG_Schock = pd.read_csv(datenpfad)  # Read the LMU dataset

datenpfad = "D:/AG_USB/Mohamed_AG_Schock/Datasets/eICU_last.csv"
df_eicu = pd.read_csv(datenpfad)  # Read the eICU dataset

datenpfad = "D:/AG_USB/Mohamed_AG_Schock/Datasets/MIMIC_last.csv"
df_mimic = pd.read_csv(datenpfad)  # Read the MIMIC dataset

datenpfad = 'D:/AG_USB/Mohamed_AG_Schock/Datasets/AmsterdamUMCDB_last.csv'
df_amsterdam = pd.read_csv(datenpfad)  # Read the Amsterdam dataset

datenpfad = 'D:/AG_USB/Mohamed_AG_Schock/Datasets/HiRID_last.csv'
df_hirid = pd.read_csv(datenpfad)  # Read the HiRID dataset

datenpfad = 'D:/AG_USB/Mohamed_AG_Schock/Datasets/SICdb_last.csv'
df_sicdb = pd.read_csv(datenpfad)  # Read the SICDB dataset


In [9]:
#load the needed datasets

datenpfad = '/home/scherer/Mohamed_Ali/Mohamed_Ali/Mohamed_AG_Schock/Datasets/lmu_big_last.csv'
df_AG_Schock = pd.read_csv(datenpfad)

datenpfad = "/home/scherer/Mohamed_Ali/Mohamed_Ali/Mohamed_AG_Schock/Datasets/eICU_last.csv"
df_eicu = pd.read_csv(datenpfad)

datenpfad = "/home/scherer/Mohamed_Ali/Mohamed_Ali/Mohamed_AG_Schock/Datasets/MIMIC_last.csv"
df_mimic = pd.read_csv(datenpfad)

datenpfad = '/home/scherer/Mohamed_Ali/Mohamed_Ali/Mohamed_AG_Schock/Datasets/AmsterdamUMCDB_last.csv'
df_amsterdam = pd.read_csv(datenpfad)

datenpfad = '/home/scherer/Mohamed_Ali/Mohamed_Ali/Mohamed_AG_Schock/Datasets/HiRID_last.csv'
df_hirid = pd.read_csv(datenpfad)

datenpfad = '/home/scherer/Mohamed_Ali/Mohamed_Ali/Mohamed_AG_Schock/Datasets/SICdb_last.csv'
df_sicdb = pd.read_csv(datenpfad)


In [3]:
df_AG_Schock.rename(columns={'Pao2weightedmean': 'PO2WeightedMean'}, inplace=True)
df_AG_Schock.rename(columns={'PaCO2weightedmean': 'PCO2WeightedMean'}, inplace=True)
df_AG_Schock.rename(columns={'UreaWeightedMean': 'ureaWeightedMean'}, inplace=True)
df_AG_Schock.rename(columns={'HCO3WeightedMean1': 'HCO3WeightedMean'}, inplace=True)
df_AG_Schock.rename(columns={'PlateletsWeightedMean': 'plateletsWeightedMean'}, inplace=True)
df_AG_Schock.rename(columns={'LeukocytesWeightedMean': 'leucoWeightedMean'}, inplace=True)
df_AG_Schock.rename(columns={'DaysTillHospitalStay_end': 'DaystillDeath'}, inplace=True)
df_AG_Schock.rename(columns={'gluWeightedMean1': 'gluWeightedMean'}, inplace=True)
df_AG_Schock.loc[df_AG_Schock['Death_inHospitalStay'] == 0, 'DaystillDeath'] = np.nan

df_AG_Schock['HighestLactate'] = df_AG_Schock.apply(lambda row: max(row['HighestLactate_0_12'], row['HighestLactate_12_24']), axis=1)
df_AG_Schock['LowestpH'] = df_AG_Schock.apply(lambda row: min(row['LowestpH_0_12'], row['LowestpH_12_24']), axis=1)


In [4]:
df_AG_Schock = filter_df(df_AG_Schock)
df_eicu = filter_df(df_eicu)
df_mimic = filter_df(df_mimic)
df_amsterdam = filter_df(df_amsterdam)
df_hirid = filter_df(df_hirid)
df_sicdb = filter_df(df_sicdb)
df_mimic_iv = filter_df(df_mimic_iv)


In [5]:
#df_AG_Schock = df_AG_Schock.loc[df_AG_Schock['ID'] <= 4131]
df_AG_Schock['Datum'] = pd.to_datetime(df_AG_Schock['Datum'], errors='coerce')
df_AG_Schock = df_AG_Schock.loc[df_AG_Schock['Datum'] <= pd.to_datetime('2024-03-01 01:01:01')]
df_AG_Schock = clean_data_small(df_AG_Schock)

criteria = (
    (df_AG_Schock.groupby('ID')['MCS'].max() == 1) |  # MCS irgendwann 1
    (df_AG_Schock.groupby('ID')['Vasopressin_full_dose'].max() > 0) |
    (df_AG_Schock.groupby('ID')['Dobutamine_full_dose'].max() > 0) |
    (df_AG_Schock.groupby('ID')['Norepinephrine_full_dose'].max() > 0) |
    (df_AG_Schock.groupby('ID')['Epinephrine_full_dose'].max() > 0)
)

valid_ids = criteria[criteria].index

df_AG_Schock = df_AG_Schock[df_AG_Schock['ID'].isin(valid_ids)]


In [ ]:
# -----------------------
# Data Cleaning & Preprocessing
# -----------------------

# Define safe ranges for common lab values and vital signs
def clean_outliers(df, column, lower=None, upper=None, multiply=None, rename=None):
    if multiply:
        df[column] = df[column] * multiply
    if rename:
        df.rename(columns={column: rename}, inplace=True)
    if lower is not None:
        df.loc[df[column] < lower, column] = np.nan
    if upper is not None:
        df.loc[df[column] > upper, column] = np.nan


# --- EICU Dataset ---
clean_outliers(df_eicu, 'RRDiaWeightedMeanValue', 30, 130)
clean_outliers(df_eicu, 'sO2WeightedMeanValue', 30, 100)
clean_outliers(df_eicu, 'PHWeightedMean', 6.9, 7.7)
clean_outliers(df_eicu, 'LactateWeightedMean', upper=50)
clean_outliers(df_eicu, 'gluWeightedMean', 50, 600)
clean_outliers(df_eicu, 'HCO3WeightedMean', 9)
clean_outliers(df_eicu, 'CreaWeightedMean', rename='CreatinineWeightedMean')
clean_outliers(df_eicu, 'TempWeightedMean', 20, 50)
clean_outliers(df_eicu, 'CreatinineWeightedMean', upper=40)
clean_outliers(df_eicu, 'Height', lower=20)
clean_outliers(df_eicu, 'HighestLactate', upper=80)
clean_outliers(df_eicu, 'LowestpH', lower=5)

# --- SICDB Dataset ---
clean_outliers(df_sicdb, 'RRSysWeightedMeanValue', 40, 190)
clean_outliers(df_sicdb, 'RRDiaWeightedMeanValue', 25, 140)
clean_outliers(df_sicdb, 'PHWeightedMean', lower=5, upper=7.7)
clean_outliers(df_sicdb, 'TempWeightedMean', 20, 40)
clean_outliers(df_sicdb, 'Height', 20)
clean_outliers(df_sicdb, 'Weight', 10)
clean_outliers(df_sicdb, 'LowestpH', 5)
clean_outliers(df_sicdb, 'CreatinineWeightedMean', upper=40)
clean_outliers(df_sicdb, 'HighestLactate', upper=80)

# --- Amsterdam Dataset ---
clean_outliers(df_amsterdam, 'gluWeightedMean', multiply=18.0182)
clean_outliers(df_amsterdam, 'ureaWeightedMean', multiply=6.006)
clean_outliers(df_amsterdam, 'CreatinineWeightedMean', multiply=0.011)
clean_outliers(df_amsterdam, 'HCO3WeightedMean', upper=50)
clean_outliers(df_amsterdam, 'RRSysWeightedMeanValue', 40, 200)
clean_outliers(df_amsterdam, 'RRDiaWeightedMeanValue', 30, 130)
clean_outliers(df_amsterdam, 'sO2WeightedMeanValue', upper=100)
clean_outliers(df_amsterdam, 'PHWeightedMean', 6.9, 7.7)
clean_outliers(df_amsterdam, 'PO2WeightedMean', upper=400)
clean_outliers(df_amsterdam, 'PCO2WeightedMean', upper=100)
clean_outliers(df_amsterdam, 'HRWeightedMean', upper=200)
clean_outliers(df_amsterdam, 'TempWeightedMean', 20, 45)
clean_outliers(df_amsterdam, 'ClWeightedMean', lower=10)
clean_outliers(df_amsterdam, 'LowestpH', lower=5)
clean_outliers(df_amsterdam, 'CreatinineWeightedMean', upper=40)
clean_outliers(df_amsterdam, 'Height', lower=20)
clean_outliers(df_amsterdam, 'HighestLactate', upper=80)

# --- MIMIC Dataset ---
clean_outliers(df_mimic, 'gluWeightedMean', 50, 600)
clean_outliers(df_mimic, 'HCO3WeightedMean', 9)
clean_outliers(df_mimic, 'NaWeightedMean', lower=0)
clean_outliers(df_mimic, 'TempWeightedMean', 20, 50)
clean_outliers(df_mimic, 'CreatinineWeightedMean', upper=40)
clean_outliers(df_mimic, 'Height', lower=20)
clean_outliers(df_mimic, 'HighestLactate', upper=80)
clean_outliers(df_mimic, 'LowestpH', lower=5)

# --- HiRID Dataset ---
clean_outliers(df_hirid, 'ureaWeightedMean', multiply=6.006)
clean_outliers(df_hirid, 'HBWeightedMean', multiply=0.1)
clean_outliers(df_hirid, 'CreatinineWeightedMean', multiply=0.011)
clean_outliers(df_hirid, 'TempWeightedMean', 20, 44)
clean_outliers(df_hirid, 'NaWeightedMean', upper=200)
clean_outliers(df_hirid, 'KWeightedMean', upper=10)
clean_outliers(df_hirid, 'gluWeightedMean', multiply=18.0182)
clean_outliers(df_hirid, 'gluWeightedMean', upper=600)
clean_outliers(df_hirid, 'HCO3WeightedMean', lower=10)
clean_outliers(df_hirid, 'sO2WeightedMeanValue', upper=100)
clean_outliers(df_hirid, 'CreatinineWeightedMean', upper=40)
clean_outliers(df_hirid, 'TempWeightedMean', lower=20)
clean_outliers(df_hirid, 'Height', lower=20)
clean_outliers(df_hirid, 'HighestLactate', upper=80)
clean_outliers(df_hirid, 'LowestpH', lower=5)

# --- AG-Schock Dataset ---
clean_outliers(df_AG_Schock, 'LactateWeightedMean', lower=0)
clean_outliers(df_AG_Schock, 'gluWeightedMean', lower=60)
clean_outliers(df_AG_Schock, 'TempWeightedMean', 20, 50)
clean_outliers(df_AG_Schock, 'CreatinineWeightedMean', upper=40)
clean_outliers(df_AG_Schock, 'Height', lower=20)
clean_outliers(df_AG_Schock, 'HighestLactate', upper=80)
clean_outliers(df_AG_Schock, 'LowestpH', lower=5)


In [ ]:
# Save the cleaned dataframes to new CSV files
df_AG_Schock.to_csv("lmu_final_training.csv", index=False)  # Save the cleaned LMU dataset
df_eicu.to_csv("eicu_final_training.csv", index=False)  # Save the cleaned eICU dataset
df_mimic.to_csv("mimic_final_training.csv", index=False)  # Save the cleaned MIMIC dataset
df_amsterdam.to_csv("amsterdam_final_training.csv", index=False)  # Save the cleaned Amsterdam dataset
df_hirid.to_csv("hirid_final_training.csv", index=False)  # Save the cleaned HiRID dataset
df_sicdb.to_csv("SICdb_final_training.csv", index=False)  # Save the cleaned SICDB dataset